<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# FLOPS 분석(FLOPS Analysis)

- FLOPS(Floating Point Operations Per Second)는 수행되는 부동소수점 연산의 개수를 계산하여 신경망 모델의 계산 복잡도를 측정합니다
- 높은 FLOPS는 더 집약적인 계산과 에너지 소비를 나타냅니다

In [ ]:
# pip install -r requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = [
    "thop",
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

thop version: 0.1.1-2209072238
torch version: 2.4.1+cu121


&nbsp;
# 고정된 배치 크기로 간단한 벤치마크

- 순전파(forward pass)만

In [ ]:
import torch
from thop import profile

# 설치 지침은 다음을 참조하세요:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
from llms_from_scratch.ch04 import GPTModel


BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘 크기
    "context_length": 1024,  # 컨텍스트 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # 쿼리-키-값 편향
}

model_configs = {
    "gpt-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 2
input_tensor = torch.randint(0, 50257, (batch_size, 1024)).to(device)

for size in model_configs:
    BASE_CONFIG.update(model_configs[size])

    model = GPTModel(BASE_CONFIG).bfloat16()
    model.to(device)

    # MACS = 곱셈-누적 연산(multiply-accumulate operations)
    # MACS는 일반적으로 두 개의 FLOPS로 계산됩니다 (하나의 곱셈과 하나의 누적)
    macs, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops = 2*macs
    print(f"{size:18}: {flops:.1e} FLOPS")

    del model
    torch.cuda.empty_cache()

gpt-small (124M)  : 5.1e+11 FLOPS
gpt-medium (355M) : 1.4e+12 FLOPS
gpt-large (774M)  : 3.2e+12 FLOPS
gpt-xl (1558M)    : 6.4e+12 FLOPS


&nbsp;
# 자동 배치 크기 찾기로 간단한 벤치마크

- 순전파(forward pass)만

In [ ]:
for size in model_configs:
    print(f"\nProcessing {size}")
    config = BASE_CONFIG.copy()
    config.update(model_configs[size])

    min_batch_size = 1
    max_batch_size = None
    max_possible_batch_size = 4096

    while min_batch_size <= max_possible_batch_size:
        batch_size = (min_batch_size + max_possible_batch_size) // 2
        try:
            input_tensor = torch.randint(
                0, config["vocab_size"],
                (batch_size, config["context_length"]),
                device=device
            )

            model = GPTModel(config).bfloat16().to(device)

            # MACS = 곱셈-누적 연산(multiply-accumulate operations)
            # MACS는 일반적으로 두 개의 FLOPS로 계산됩니다 (하나의 곱셈과 하나의 누적)
            macs, params = profile(model, inputs=(input_tensor,), verbose=False)
            flops = 2 * macs
            print(f"  Batch size {batch_size}: {flops:.1e} FLOPS")

            # 성공하면 더 큰 배치 크기를 시도
            min_batch_size = batch_size + 1
            max_batch_size = batch_size

            # 정리
            del model, input_tensor
            torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                # 더 작은 배치 크기 시도
                max_possible_batch_size = batch_size - 1

                # 정리
                try:
                    del model, input_tensor
                    torch.cuda.empty_cache()
                except NameError:
                    pass
            else:
                raise e


Processing gpt-small (124M)
  Batch size 256: 6.5e+13 FLOPS
  Batch size 384: 9.7e+13 FLOPS
  Batch size 388: 9.8e+13 FLOPS
  Batch size 389: 9.8e+13 FLOPS

Processing gpt-medium (355M)
  Batch size 256: 1.9e+14 FLOPS
  Batch size 260: 1.9e+14 FLOPS
  Batch size 262: 1.9e+14 FLOPS
  Batch size 263: 1.9e+14 FLOPS

Processing gpt-large (774M)
  Batch size 256: 4.0e+14 FLOPS

Processing gpt-xl (1558M)
  Batch size 128: 4.1e+14 FLOPS
  Batch size 136: 4.3e+14 FLOPS
  Batch size 140: 4.5e+14 FLOPS
  Batch size 142: 4.5e+14 FLOPS
  Batch size 143: 4.6e+14 FLOPS


&nbsp;
# 자동 배치 크기 찾기와 모델 FLOP 활용률(Model FLOP Utilization, MFU)을 포함한 벤치마크

- [PaLM 논문](https://arxiv.org/abs/2204.02311)의 모델 FLOPS 활용률(Model FLOPs Utilization, MFU) 설명

> 우리는 구현에 독립적이고 시스템 효율성의 더 깔끔한 비교를 가능하게 하는 효율성을 위한 새로운 지표인 모델 FLOPS 활용률(model FLOPs utilization, MFU)을 제안합니다. 이는 최대 FLOPS에서 작동하는 시스템의 이론적 최대 처리량에 대한 관찰된 처리량(초당 토큰 수)의 비율입니다. 중요한 점은 "이론적 최대" 처리량이 순전파+역전파 패스를 계산하는 데 필요한 연산만을 고려하며, 재실체화(rematerialization)는 고려하지 않는다는 것입니다.


$$\text{MFU} = \frac{\text{관찰된 초당 토큰 수}}{\text{이론적 최대 초당 토큰 수}}$$

여기서

$$\text{이론적 최대 초당 토큰 수} = \frac{\text{최대 초당 FLOPS}}{\text{토큰당 총 FLOPS}}$$

그리고

$$\text{초당 토큰 수} = \frac{\text{배치 크기} \times \text{시퀀스 길이}}{\text{총 시간}}$$

- 순전파와 역전파

In [ ]:
# GPU 제조업체에서 제공하는 이론적 최대 초당 FLOPS

flops_per_second = {
    # https://www.techpowerup.com/gpu-specs/h100-pcie-80-gb.c3899
    "H100": {
        torch.float32: 51.22e12,  # NVIDIA H100에서 FP32용 51.22 TFLOPs
        torch.float16: 204.9e12,  # NVIDIA H100에서 FP16용 204.9 TFLOPs
        torch.bfloat16: 204.9e12
    },
    # https://www.techpowerup.com/gpu-specs/l4.c4091
    "L4": {
        torch.float32: 30.29e12,  # NVIDIA L4에서 FP32용 30.29 TFLOPs
        torch.float16: 30.29e12,  # NVIDIA L4에서 FP16용 30.29 TFLOPs
        torch.bfloat16: 30.29e12
    },
    # https://www.techpowerup.com/gpu-specs/tesla-t4.c3316
    "T4": {
        torch.float32: 8.1e12,  # NVIDIA T4에서 FP32용 8.1 TFLOPs
        torch.float16: 65.13e12,  # NVIDIA T4에서 FP16용 65.13 TFLOPs
        torch.bfloat16: 65.13e12
    },
    # https://www.techpowerup.com/gpu-specs/a10g.c3798
    "A10G": {
        torch.float32: 31.52e12,  # NVIDIA A10G에서 FP32용 31.52 TFLOPs
        torch.float16: 31.52e12,  # NVIDIA A10G에서 FP16용 31.52 TFLOPs
        torch.bfloat16: 31.52e12
    },
    # https://www.techpowerup.com/gpu-specs/a100-pcie-40-gb.c3623
    "A100": {
        torch.float32: 19.49e12,  # NVIDIA A100에서 FP32용 19.49 TFLOPs
        torch.float16: 77.97e12,  # NVIDIA A100에서 FP16용 77.97 TFLOPs
        torch.bfloat16: 77.97e12
    },
    # https://www.techpowerup.com/gpu-specs/geforce-rtx-3080.c3621
    "RTX_3080": {
        torch.float32: 29.77e12,  # NVIDIA RTX 3080에서 FP32용 29.77 TFLOPs
        torch.float16: 29.77e12,  # NVIDIA RTX 3080에서 FP16용 29.77 TFLOPs
        torch.bfloat16: 29.77e12
    },
    # https://www.techpowerup.com/gpu-specs/geforce-rtx-3090.c3622
    "RTX_3090": {
        torch.float32: 35.58e12,  # NVIDIA RTX 3090에서 FP32용 35.58 TFLOPs
        torch.float16: 35.58e12,  # NVIDIA RTX 3090에서 FP16용 35.58 TFLOPs
        torch.bfloat16: 35.58e12
    }
}

In [ ]:
import time

def get_gpu_model(flops_per_second_dict):
    device_name = torch.cuda.get_device_name(0)
    for model in flops_per_second_dict.keys():
        if model in device_name:
            return model
    return "Unknown"  # 일치하는 모델이 없으면 기본값


gpu_model = get_gpu_model(flops_per_second)
print("GPU Model:", gpu_model)

if gpu_model != "Unknown":

    for size in model_configs:
        print(f"\nProcessing {size}")
        config = BASE_CONFIG.copy()
        config.update(model_configs[size])

        min_batch_size = 1
        max_batch_size = None
        max_possible_batch_size = 4096

        while min_batch_size <= max_possible_batch_size:
            batch_size = (min_batch_size + max_possible_batch_size) // 2
            try:
                input_tensor = torch.randint(
                    0, config["vocab_size"],
                    (batch_size, config["context_length"]),
                    device=device
                )

                model = GPTModel(config).bfloat16().to(device)
                model.train()

                # 시간 측정 시작
                torch.cuda.synchronize()
                start_time = time.time()

                # 순전파 & 역전파
                output = model(input_tensor)
                loss = output.sum()  # 더미 손실 계산
                loss.backward()

                # 시간 측정 종료
                torch.cuda.synchronize()
                end_time = time.time()

                total_time_seconds = end_time - start_time

                # 순전파의 FLOPS 계산
                macs, params = profile(model, inputs=(input_tensor,), verbose=False)
                flops_forward = 2 * macs  # 하나의 MAC이 두 개의 FLOPS와 같다고 가정

                # 역전파의 FLOPS 추정 (일반적으로 순전파 FLOPS의 2배)
                flops_backward = 2 * flops_forward

                # 순전파 + 역전파의 총 FLOPS
                total_flops = flops_forward + flops_backward  # 또는 total_flops = flops_forward * 3

                data_type = next(model.parameters()).dtype
                max_flops_per_second = flops_per_second[gpu_model].get(data_type, 0)

                # 초당 토큰 수 계산
                tokens_processed = batch_size * config["context_length"]
                tokens_per_second = tokens_processed / total_time_seconds

                # 토큰당 FLOPS 계산
                flops_per_token = total_flops / tokens_processed

                # 이론적 최대 초당 토큰 수 계산
                if flops_per_token > 0:
                    theoretical_max_tokens_per_second = max_flops_per_second / flops_per_token
                else:
                    theoretical_max_tokens_per_second = 0  # 0으로 나누기 방지

                # MFU 계산
                if theoretical_max_tokens_per_second > 0:
                    mfu = tokens_per_second / theoretical_max_tokens_per_second
                else:
                    mfu = 0  # 0으로 나누기 방지

                print(f"  Batch size {batch_size}: Tokens/sec: {tokens_per_second:.2f}, MFU: {mfu:.4f}")

                # 성공하면 더 큰 배치 크기를 시도
                min_batch_size = batch_size + 1
                max_batch_size = batch_size

                # 정리
                del model, input_tensor, output, loss
                torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    # 더 작은 배치 크기 시도
                    max_possible_batch_size = batch_size - 1

                    # 정리
                    try:
                        del model, input_tensor
                        torch.cuda.empty_cache()
                    except NameError:
                        pass
                else:
                    raise e

else:
    print("알 수 없는 GPU 모델입니다. GPU 정보로 flops_per_second 딕셔너리를 업데이트해 주세요.")

GPU Model: A100

Processing gpt-small (124M)
  Batch size 16: Tokens/sec: 34248.82, MFU: 0.3256
  Batch size 24: Tokens/sec: 62568.34, MFU: 0.5948

Processing gpt-medium (355M)
  Batch size 4: Tokens/sec: 20159.93, MFU: 0.5483
  Batch size 6: Tokens/sec: 21717.66, MFU: 0.5907
  Batch size 7: Tokens/sec: 22536.25, MFU: 0.6130

Processing gpt-large (774M)
  Batch size 8: Tokens/sec: 12465.21, MFU: 0.7406

Processing gpt-xl (1558M)
  Batch size 4: Tokens/sec: 6779.92, MFU: 0.8113


- 1.0의 값이 최상입니다 (100%와 동일)
- 배치 크기가 이전보다 작은 것을 주목하세요. 여기서는 더 메모리 집약적인 역전파도 수행하기 때문입니다